<a href="https://colab.research.google.com/github/muazzamasim04-ops/evolvix-aiml-internship-muazzamasim/blob/main/Week_3_Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install compatible packages (without breaking torch/torchvision)
!pip install -q "transformers[torch]" datasets evaluate scikit-learn accelerate

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# ---------------------------------------------------------
# 2. Load Dataset
# ---------------------------------------------------------
dataset = load_dataset("stanfordnlp/imdb")

# Select sub-samples for fast runtime (1-2 minutes on GPU)
train_data = dataset['train'].shuffle(seed=42).select(range(4000))
test_data = dataset['test'].shuffle(seed=42).select(range(1000))

X_train, y_train = train_data['text'], train_data['label']
X_test, y_test = test_data['text'], test_data['label']

label_map = {0: "Negative", 1: "Positive"}

# ---------------------------------------------------------
# 3. Classical Baseline: TF-IDF + Logistic Regression
# ---------------------------------------------------------
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

preds_lr = clf.predict(X_test_vec)
probs_lr = clf.predict_proba(X_test_vec)

lr_acc = accuracy_score(y_test, preds_lr)
lr_f1 = f1_score(y_test, preds_lr)

print(f"Classical Model (TF-IDF + Logistic Regression):")
print(f"  Accuracy: {lr_acc:.4f}")
print(f"  F1 Score: {lr_f1:.4f}\n")

# ---------------------------------------------------------
# 4. Pretrained Transformer: Fine-Tuning DistilBERT
# ---------------------------------------------------------
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_tok = train_data.map(tokenize_fn, batched=True)
test_tok = test_data.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=True, # GPU Acceleration
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

# Predict with Transformer
raw_preds = trainer.predict(test_tok)
logits = torch.tensor(raw_preds.predictions)
probs_tf = torch.softmax(logits, dim=-1).numpy()
preds_tf = np.argmax(probs_tf, axis=-1)

tf_acc = accuracy_score(y_test, preds_tf)
tf_f1 = f1_score(y_test, preds_tf)

print(f"\nTransformer Model (DistilBERT):")
print(f"  Accuracy: {tf_acc:.4f}")
print(f"  F1 Score: {tf_f1:.4f}\n")

# ---------------------------------------------------------
# 5. Model Comparison & Example Predictions
# ---------------------------------------------------------
sample_indices = [5, 12, 45, 89]

print("--- Example Predictions ---")
for idx in sample_indices:
    text = X_test[idx][:120] + "..."
    true_label = label_map[y_test[idx]]

    lr_pred = label_map[preds_lr[idx]]
    lr_conf = np.max(probs_lr[idx])

    tf_pred = label_map[preds_tf[idx]]
    tf_conf = np.max(probs_tf[idx])

    print(f"\nText: {text}")
    print(f"True Sentiment: {true_label}")
    print(f"Classical LR:  Pred = {lr_pred} (Confidence: {lr_conf:.2%})")
    print(f"DistilBERT:    Pred = {tf_pred} (Confidence: {tf_conf:.2%})")

ModuleNotFoundError: Could not import module 'Trainer'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

In [1]:
# 1. Clear old conflicting torchvision and install compatible packages
!pip uninstall -y torchvision
!pip install -q "transformers[torch]" datasets evaluate scikit-learn accelerate

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# ---------------------------------------------------------
# 2. Load Dataset
# ---------------------------------------------------------
dataset = load_dataset("stanfordnlp/imdb")

# Select sub-samples for fast runtime (1-2 minutes on GPU)
train_data = dataset['train'].shuffle(seed=42).select(range(4000))
test_data = dataset['test'].shuffle(seed=42).select(range(1000))

X_train, y_train = train_data['text'], train_data['label']
X_test, y_test = test_data['text'], test_data['label']

label_map = {0: "Negative", 1: "Positive"}

# ---------------------------------------------------------
# 3. Classical Baseline: TF-IDF + Logistic Regression
# ---------------------------------------------------------
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

preds_lr = clf.predict(X_test_vec)
probs_lr = clf.predict_proba(X_test_vec)

lr_acc = accuracy_score(y_test, preds_lr)
lr_f1 = f1_score(y_test, preds_lr)

print(f"Classical Model (TF-IDF + Logistic Regression):")
print(f"  Accuracy: {lr_acc:.4f}")
print(f"  F1 Score: {lr_f1:.4f}\n")

# ---------------------------------------------------------
# 4. Pretrained Transformer: Fine-Tuning DistilBERT
# ---------------------------------------------------------
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_tok = train_data.map(tokenize_fn, batched=True)
test_tok = test_data.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=True,  # Fast execution on T4 GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

# Predict with Transformer
raw_preds = trainer.predict(test_tok)
logits = torch.tensor(raw_preds.predictions)
probs_tf = torch.softmax(logits, dim=-1).numpy()
preds_tf = np.argmax(probs_tf, axis=-1)

tf_acc = accuracy_score(y_test, preds_tf)
tf_f1 = f1_score(y_test, preds_tf)

print(f"\nTransformer Model (DistilBERT):")
print(f"  Accuracy: {tf_acc:.4f}")
print(f"  F1 Score: {tf_f1:.4f}\n")

# ---------------------------------------------------------
# 5. Model Comparison & Example Predictions
# ---------------------------------------------------------
sample_indices = [5, 12, 45, 89]

print("--- Example Predictions ---")
for idx in sample_indices:
    text = X_test[idx][:120] + "..."
    true_label = label_map[y_test[idx]]

    lr_pred = label_map[preds_lr[idx]]
    lr_conf = np.max(probs_lr[idx])

    tf_pred = label_map[preds_tf[idx]]
    tf_conf = np.max(probs_tf[idx])

    print(f"\nText: {text}")
    print(f"True Sentiment: {true_label}")
    print(f"Classical LR:  Pred = {lr_pred} (Confidence: {lr_conf:.2%})")
    print(f"DistilBERT:    Pred = {tf_pred} (Confidence: {tf_conf:.2%})")

Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.411633,0.393747,0.828000,0.838041
2,0.220125,0.407418,0.841000,0.840201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Transformer Model (DistilBERT):
  Accuracy: 0.8410
  F1 Score: 0.8402

--- Example Predictions ---

Text: After a very long time Marathi cinema has come with some good movie.This movie is one of the best Marathi movies ever ma...
True Sentiment: Positive
Classical LR:  Pred = Positive (Confidence: 56.20%)
DistilBERT:    Pred = Positive (Confidence: 98.10%)

Text: These days, writers, directors and producers are relying more and more on the "surprise" ending. The old art of bringing...
True Sentiment: Negative
Classical LR:  Pred = Negative (Confidence: 84.50%)
DistilBERT:    Pred = Negative (Confidence: 94.18%)

Text: I have done a lot of international travel, both on business and as a tourist. For both types I assure you the best advic...
True Sentiment: Positive
Classical LR:  Pred = Positive (Confidence: 75.72%)
DistilBERT:    Pred = Positive (Confidence: 93.73%)

Text: "Two Hands" is an entertaining, funny story about Australian lowlifes. The screenplay contrasts the world of fast